# Modelado Predictivo: Supervivencia en el Titanic

Este cuaderno implementa el ciclo completo de Machine Learning siguiendo las mejores prácticas:
- Formulación del problema y baselines de referencia sin fuga de datos.
- Preprocesamiento reproducible con `Pipeline` y `ColumnTransformer`.
- Comparación de modelos con validación cruzada estratificada.
- Análisis de errores, curvas de aprendizaje e interpretabilidad.

## Parte 10 — Del análisis a la predicción: formulación del problema (Pasos 39–42)

### Preparación: Carga de la capa Silver para modelado
Importamos el módulo `flows.05_ml_models` y cargamos el dataset limpio. **Nota fundamental**: la función `ml.load_dataset()` toma la edad original (`AgeOriginal` con valores nulos) en lugar de la edad imputada en el EDA, para evitar que información del conjunto de prueba contamine el entrenamiento (*Data Leakage*).

In [1]:
import sys
from pathlib import Path

# Asegura que la raíz del proyecto esté en sys.path
root = Path.cwd() if (Path.cwd() / "flows").exists() else Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ml = importlib.import_module("flows.05_ml_models")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

# Carga de Silver: X con columnas candidatas, y = Survived
X, y = ml.load_dataset()
print(f"Dimensiones de X: {X.shape}")
print(f"Tasa de supervivencia global: {y.mean():.4f}")

Dimensiones de X: (891, 12)
Tasa de supervivencia global: 0.3838


### Paso 39. Definir el problema y la métrica

Las decisiones metodológicas están documentadas en el informe `reports/problema_ml.md`:
- **Tipo de problema**: Clasificación binaria supervisada.
- **Variable objetivo**: `Survived` (1 = sobrevivió, 0 = no sobrevivió).
- **Unidad de análisis y momento de predicción**: Un pasajero individual, **al momento de embarcar**.
- **Métrica principal**: **ROC-AUC** (robusta ante el desbalance natural de clases del ~38.4% y evaluable sobre todos los umbrales).
- **Métricas secundarias**: Exactitud (*Accuracy*), Precisión, Recall y F1-Score.

### Paso 40. Auditoría de variables y fuga de datos

Auditoría de columnas candidatas para prevenir el sobreajuste y la fuga de información:
- **Excluidas**: `Survived`/`Survivor` (fuga directa), `PassengerId`/`Ticket`/`Name` (identificadores con alta cardinalidad), `NSex`/`NEmbarked`/`NCabin`/`NTitulo` (codificaciones ordinales arbitrarias, se usará One-Hot en el Pipeline).
- **Mantenidas**: Variables sociodemográficas conocidas antes de zarpar (`Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, `Embarked`, `TTitulo`, `Alone`, `Familia`).
- **Sospechosas para ablación**: `HasCabin` y `TCabin` (se evaluará si son un artefacto de registro en el Paso 61).

In [2]:
# Paso 40 — Inspección de las características seleccionadas
print("Variables candidatas por grupo:")
print(f"Numéricas:   {ml.NUM_FEATURES}")
print(f"Categóricas: {ml.CAT_FEATURES}")
print(f"Cabina:      {ml.CABIN_NUM} (num), {ml.CABIN_CAT} (cat)")

resumen_variables = pd.DataFrame({
    "Tipo": X.dtypes,
    "Valores Únicos": X.nunique(),
    "Faltantes": X.isnull().sum(),
    "% Faltantes": (X.isnull().mean() * 100).round(2)
})
display(resumen_variables)

Variables candidatas por grupo:
Numéricas:   ['Age', 'SibSp', 'Parch', 'Fare', 'Familia']
Categóricas: ['Pclass', 'Sex', 'Embarked', 'TTitulo', 'Alone']
Cabina:      ['HasCabin'] (num), ['TCabin'] (cat)


,Tipo,Valores Únicos,Faltantes,% Faltantes
Age,float64,88,177,19.87
SibSp,int64,7,0,0.00
Parch,int64,7,0,0.00
Fare,float64,248,0,0.00
Familia,int64,9,0,0.00
Pclass,int64,3,0,0.00
Sex,str,2,0,0.00
Embarked,str,3,0,0.00
TTitulo,str,4,0,0.00
Alone,str,2,0,0.00


### Paso 41. División estratificada entrenamiento / prueba

Dividimos los datos en 80% entrenamiento y 20% prueba con estratificación por `y` y semilla fija (`random_state=42`).

> **Regla de oro de evaluación**: El conjunto `X_test` e `y_test` se almacena aislado y **no se vuelve a tocar** hasta la evaluación final (Parte 16). Todas las decisiones de preprocesamiento, selección de modelos y ajuste de hiperparámetros se realizan mediante validación cruzada exclusivamente sobre `X_train` e `y_train`.

In [3]:
# Paso 41 — División estratificada Train / Test
X_train, X_test, y_train, y_test = ml.split(X, y)

print(f"Dimensiones X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Dimensiones X_test:  {X_test.shape}, y_test:  {y_test.shape}")
print(f"Tasa de supervivencia en Train: {y_train.mean():.4f}")
print(f"Tasa de supervivencia en Test:  {y_test.mean():.4f}")

# Comprobamos que la proporción se mantiene idéntica
assert abs(y_train.mean() - y_test.mean()) < 0.01, "Error: proporciones no estratificadas"
print("\n✓ Estratificación verificada exitosamente. X_test queda reservado hasta la Parte 16.")

Dimensiones X_train: (712, 12), y_train: (712,)
Dimensiones X_test:  (179, 12), y_test:  (179,)
Tasa de supervivencia en Train: 0.3834
Tasa de supervivencia en Test:  0.3855

✓ Estratificación verificada exitosamente. X_test queda reservado hasta la Parte 16.


### Paso 42. Baselines: lo mínimo que un modelo debe superar

Evaluamos dos puntos de referencia esenciales:
1. **Baseline ingenuo**: `DummyClassifier(strategy='most_frequent')` — predice siempre la clase mayoritaria (0: no sobrevive). Su exactitud refleja únicamente la prevalencia de la clase.
2. **Baseline heurístico (Regla de género)**: Predice 1 si el pasajero es mujer (`Sex == 'female'`) y 0 si es hombre (`Sex == 'male'`). Es el reflejo directo del protocolo histórico *"mujeres y niños primero"*.

In [4]:
# Paso 42 — Evaluación y comparación de baselines
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate

# 1. Baseline ingenuo evaluado con validación cruzada (5 folds) sobre train
dummy = DummyClassifier(strategy="most_frequent")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_dummy = cross_validate(dummy, X_train, y_train, cv=cv, scoring=ml.SCORING)

# 2. Baseline heurístico de regla de género sobre train
y_pred_gender = (X_train["Sex"] == "female").astype(int)

# 3. Tabla comparativa consolidada
tabla_baselines = pd.DataFrame([
    {
        "Baseline": "1. Ingenuo (Clase Mayoritaria)",
        "Exactitud": f"{cv_dummy['test_accuracy'].mean():.4f} +/- {cv_dummy['test_accuracy'].std():.4f}",
        "Precisión": f"{cv_dummy['test_precision'].mean():.4f}",
        "Recall": f"{cv_dummy['test_recall'].mean():.4f}",
        "F1-Score": f"{cv_dummy['test_f1'].mean():.4f}",
        "ROC-AUC": f"{cv_dummy['test_roc_auc'].mean():.4f}",
    },
    {
        "Baseline": "2. Heurístico (Regla de Género)",
        "Exactitud": f"{accuracy_score(y_train, y_pred_gender):.4f}",
        "Precisión": f"{precision_score(y_train, y_pred_gender):.4f}",
        "Recall": f"{recall_score(y_train, y_pred_gender):.4f}",
        "F1-Score": f"{f1_score(y_train, y_pred_gender):.4f}",
        "ROC-AUC": f"{roc_auc_score(y_train, y_pred_gender):.4f}",
    }
]).set_index("Baseline")

display(tabla_baselines)
print("\nMeta a superar: Cualquier modelo ML debe superar consistentemente a la Regla de Género (~78.7% exactitud, ~0.767 ROC-AUC).")

/home/santiago/proyectos/proyecto_ciencia_de_datos/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/santiago/proyectos/proyecto_ciencia_de_datos/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/santiago/proyectos/proyecto_ciencia_de_datos/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.


,Exactitud,Precisión,Recall,F1-Score,ROC-AUC
Baseline,,,,,
1. Ingenuo (Clase Mayoritaria),0.6166 +/- 0.0027,0.0000,0.0000,0.0000,0.5000
2. Heurístico (Regla de Género),0.7893,0.7431,0.6886,0.7148,0.7703



Meta a superar: Cualquier modelo ML debe superar consistentemente a la Regla de Género (~78.7% exactitud, ~0.767 ROC-AUC).


## Parte 11 — Preprocesamiento reproducible con `Pipeline` (Pasos 43–46)

Para evitar la **fuga de datos** (*Data Leakage*), todo el preprocesamiento (imputaciones, escalado numérico y codificación de categorías) debe aprenderse **únicamente** con datos de entrenamiento y aplicarse luego de forma idéntica a los datos de prueba o producción.

### Paso 43. Demostrar la fuga de datos

Calculamos la media de edad de las mujeres usando todos los datos ($X$) y comparamos contra la media usando únicamente los datos de entrenamiento ($X_{train}$).

In [ ]:
# Paso 43: Comparamos el promedio de edad con todos los datos vs solo train
media_edad_todos = X[X["Sex"] == "female"]["Age"].mean()
media_edad_train = X_train[X_train["Sex"] == "female"]["Age"].mean()

print(f"Media de edad mujeres (todos los datos): {media_edad_todos:.3f}")
print(f"Media de edad mujeres (solo train):     {media_edad_train:.3f}")
print(f"Diferencia:                             {abs(media_edad_todos - media_edad_train):.3f} años")

> **Conclusión sobre la fuga de datos:**
> Aunque aquí la diferencia es pequeña (~0.2 años) debido al tamaño del dataset, usar datos de prueba para calcular estadísticas de entrenamiento es un error de método grave. En datasets más pequeños o con grupos minoritarios, esta fuga distorsiona las métricas de evaluación y hace que el modelo falle al desplegarse con datos reales.

### Paso 44. Un transformador propio: imputación de edad por grupo (`GroupAgeImputer`)

Creamos en `flows/05_ml_models.py` la clase `GroupAgeImputer`. Su método `.fit()` calcula las medias de edad agrupando por `Sex` y `Pclass` con datos de entrenamiento; su método `.transform()` rellena los nulos aplicando esas medias a cualquier conjunto nuevo.

In [ ]:
# Paso 44: Probamos el GroupAgeImputer definido en flows/05_ml_models.py
imputador_edad = ml.GroupAgeImputer(group_cols=("Sex", "Pclass"))
imputador_edad.fit(X_train)

print("Promedios de edad por Sexo y Clase aprendidos solo en train:")
display(imputador_edad.group_means_.round(2))
print(f"\nMedia global de respaldo: {imputador_edad.global_mean_:.2f}")

### Paso 45. Construir el preprocesador y el pipeline completo

Ensamblamos en `flows/05_ml_models.py` la función `build_pipeline(model)`:
1. **Imputación de edad**: con `GroupAgeImputer(group_cols=('Sex', 'Pclass'))`.
2. **ColumnTransformer**:
   - **Numéricas**: imputa con la mediana y normaliza con `StandardScaler`.
   - **Categóricas**: imputa con la categoría más frecuente y aplica `OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=10)`.
3. **Estimador final**: se conecta al final del Pipeline.

In [ ]:
# Paso 45: Armamos un pipeline de prueba con regresion logistica
from sklearn.linear_model import LogisticRegression

pipeline_demo = ml.build_pipeline(LogisticRegression(max_iter=1000))
pipeline_demo

### Paso 46. Verificar el preprocesamiento

Ajustamos el preprocesador con `X_train` y transformamos `X_test`. Verificamos que no quede ningún valor nulo (`NaN`) y revisamos las columnas generadas tras la codificación *One-Hot*.

In [ ]:
# Paso 46: Verificamos que el preprocesador no deje nulos al transformar test
preprocesador = pipeline_demo[:-1].fit(X_train)
Z_test = preprocesador.transform(X_test)

print(f"Dimensiones de X_test original:    {X_test.shape}")
print(f"Dimensiones de Z_test procesado:   {Z_test.shape}")
print(f"¿Tiene valores nulos (NaN)?:       {np.isnan(Z_test).any()}")

# Nombres de las columnas finales tras One-Hot Encoding
nombres_cols = preprocesador["prep"].get_feature_names_out()
print(f"\nTotal de variables generadas: {len(nombres_cols)}")
print("Primeras 10 variables:")
for col in nombres_cols[:10]:
    print(f"  - {col}")

## Parte 12 — Modelos candidatos y validación cruzada (Pasos 47–50)

Comparamos diferentes familias de algoritmos usando **validación cruzada estratificada de 5 pliegues** exclusivamente sobre `X_train`. Esto nos permite estimar el desempeño real y la estabilidad de cada modelo sin tocar nunca el conjunto de prueba (`X_test`).

### Paso 47. Definir los modelos candidatos

Evaluamos 6 modelos representativos:
1. **Dummy**: Baseline ingenuo (más frecuente).
2. **Regresión Logística (`logreg`)**: Modelo lineal interpretable.
3. **Árbol de Decisión (`tree`)**: Reglas de decisión simples.
4. **Random Forest (`forest`)**: Ensamble de bagging con 300 árboles.
5. **HistGradientBoosting (`hgb`)**: Ensamble de boosting de alto rendimiento.
6. **KNN (`knn`)**: Modelo basado en distancias a los vecinos más cercanos.

In [ ]:
# Paso 47: Consultamos el catalogo de modelos candidatos con semilla fija
modelos_candidatos = ml.get_candidates(random_state=42)

print("Modelos listos para competir:")
for nombre, estimador in modelos_candidatos.items():
    print(f"  - {nombre:8s}: {estimador.__class__.__name__}")

### Paso 48. Validación cruzada de todos los candidatos

Ejecutamos la función `ml.cross_validate_candidates()` que entrena el pipeline completo para cada modelo sobre los 5 pliegues y registra la media y desviación estándar de cada métrica.

In [ ]:
# Paso 48: Evaluamos todos los modelos con validación cruzada sobre X_train
import warnings

import pandas as pd

# Aseguramos que X_train e y_train estén disponibles si la celda se ejecuta individualmente
if "X_train" not in globals() or "y_train" not in globals():
    X, y = ml.load_dataset()
    X_train, X_test, y_train, y_test = ml.split(X, y)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    tabla_cv = ml.cross_validate_candidates(X_train, y_train)

# Tabla resumen ordenada por la métrica principal (ROC-AUC)
resumen_cv = pd.DataFrame({
    "Modelo": tabla_cv["model"],
    "ROC-AUC (Val)": tabla_cv["test_roc_auc_mean"].round(4).astype(str) + " +/- " + tabla_cv["test_roc_auc_std"].round(4).astype(str),
    "Exactitud (Val)": tabla_cv["test_accuracy_mean"].round(4).astype(str) + " +/- " + tabla_cv["test_accuracy_std"].round(4).astype(str),
    "F1-Score (Val)": tabla_cv["test_f1_mean"].round(4).astype(str),
    "ROC-AUC (Train)": tabla_cv["train_roc_auc_mean"].round(4).astype(str),
    "Brecha Sobreajuste": tabla_cv["gap_roc_auc"].round(4).astype(str)
}).set_index("Modelo")

display(resumen_cv)


### Paso 49. Leer los resultados e inspeccionar sobreajuste

Analizamos visualmente la distribución del ROC-AUC en los pliegues y la brecha entre entrenamiento y validación (*Train - Val*):

In [ ]:
# Paso 49: Boxplot comparativo de ROC-AUC por pliegues
from sklearn.model_selection import StratifiedKFold, cross_validate

if "modelos_candidatos" not in globals():
    modelos_candidatos = ml.get_candidates(42)
if "tabla_cv" not in globals():
    tabla_cv = ml.cross_validate_candidates(X_train, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
datos_pliegues = []

for nombre, modelo in modelos_candidatos.items():
    pipe = ml.build_pipeline(modelo)
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    for puntaje in res["test_score"]:
        datos_pliegues.append({"Modelo": nombre, "ROC-AUC": puntaje})

df_pliegues = pd.DataFrame(datos_pliegues)

plt.figure(figsize=(9, 5))
sns.boxplot(data=df_pliegues, x="Modelo", y="ROC-AUC", hue="Modelo", palette="Set2", legend=False)
plt.title("Comparación de Modelos: Distribución de ROC-AUC en 5 Pliegues")
plt.ylabel("ROC-AUC")
plt.xlabel("Modelo")
plt.tight_layout()
plt.show()

# Gráfico de barras con la brecha de sobreajuste
plt.figure(figsize=(9, 4))
plt.barh(tabla_cv["model"], tabla_cv["gap_roc_auc"], color="coral")
plt.title("Brecha de Sobreajuste en ROC-AUC (Train - Val)")
plt.xlabel("Diferencia (a mayor brecha, mayor sobreajuste)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


### Paso 50. Elegir finalistas con criterio

Con base en la validación cruzada y el análisis de sobreajuste, seleccionamos dos finalistas (documentados en `reports/problema_ml.md`):

1. **Finalista 1: `hgb` (HistGradientBoosting)**:
   - **Mejor desempeño global**: ROC-AUC de **0.8867** y exactitud de **82.02 %**.
   - Captura relaciones no lineales e interacciones complejas vistas en el EDA (como Clase × Género).

2. **Finalista 2: `logreg` (Regresión Logística)**:
   - **Máxima estabilidad**: Brecha de sobreajuste prácticamente nula (**0.0062** entre Train y Val).
   - **Interpretabilidad total**: Sus coeficientes son directamente explicables para el negocio asegurador.

> *Random Forest (`forest`)* queda en segundo plano porque sobreajusta en entrenamiento (Train AUC 0.998 frente a 0.866 en validación) sin superar a HistGradientBoosting.

## Parte 13 — Evaluación en profundidad (Pasos 51–55)

Analizamos a fondo el comportamiento de los modelos finalistas mediante **predicciones fuera de muestra** (*out-of-fold*). En cada pliegue de validación cruzada, el modelo predice sobre pasajeros que nunca vio en entrenamiento, lo que nos permite evaluar matrices de confusión, calibración y curvas de error sin tocar el conjunto de prueba (`X_test`).

### Paso 51. Predicciones fuera de muestra y matriz de confusión

Comparamos la matriz de confusión y el reporte de clasificación de **HistGradientBoosting** y **Regresión Logística** con umbral estándar de 0.50:

In [ ]:
# Paso 51: Obtenemos predicciones fuera de muestra para los finalistas
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import cross_val_predict

# 1. Predicciones de probabilidad para HistGradientBoosting
pipe_hgb = ml.build_pipeline(modelos_candidatos["hgb"])
proba_hgb = cross_val_predict(pipe_hgb, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
pred_hgb = (proba_hgb >= 0.5).astype(int)

# 2. Predicciones de probabilidad para Regresion Logistica
pipe_logreg = ml.build_pipeline(modelos_candidatos["logreg"])
proba_logreg = cross_val_predict(pipe_logreg, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
pred_logreg = (proba_logreg >= 0.5).astype(int)

# 3. Graficamos las matrices de confusion
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(y_train, pred_hgb, ax=axes[0], cmap="Blues")
axes[0].set_title("Matriz de Confusión: HistGradientBoosting")

ConfusionMatrixDisplay.from_predictions(y_train, pred_logreg, ax=axes[1], cmap="Blues")
axes[1].set_title("Matriz de Confusión: Regresión Logística")
plt.tight_layout()
plt.show()

print("Reporte de clasificación detallado (HistGradientBoosting):")
print(classification_report(y_train, pred_hgb, target_names=["No sobrevive", "Sobrevive"]))

### Paso 52. Curvas ROC y de Precisión-Recall

Comparamos la curva ROC (sensibilidad vs 1-especificidad) y la curva Precisión-Recall. Esta última es muy informativa en problemas con desbalance moderado porque se enfoca exclusivamente en la clase positiva (sobrevivientes).

In [ ]:
# Paso 52: Curvas ROC y Precision-Recall comparativas
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva ROC en el primer eje
RocCurveDisplay.from_predictions(y_train, proba_hgb, name="HistGradientBoosting", ax=axes[0])
RocCurveDisplay.from_predictions(y_train, proba_logreg, name="Regresión Logística", ax=axes[0])
axes[0].plot([0, 1], [0, 1], "k--", label="Azar (AUC = 0.50)")
axes[0].set_title("Curva ROC (Receiver Operating Characteristic)")
axes[0].legend()

# Curva Precision-Recall en el segundo eje
PrecisionRecallDisplay.from_predictions(y_train, proba_hgb, name="HistGradientBoosting", ax=axes[1])
PrecisionRecallDisplay.from_predictions(y_train, proba_logreg, name="Regresión Logística", ax=axes[1])
axes[1].set_title("Curva Precisión-Recall")
axes[1].legend()

plt.tight_layout()
plt.show()

### Paso 53. Calibración y elección del umbral de decisión

Evaluamos la **calibración de probabilidades** (si predice 70%, ¿sobrevive realmente cerca del 70%?) usando el *Brier Score* y buscamos el umbral que maximiza el puntaje $F1$.

In [ ]:
# Paso 53: Calibracion de probabilidades y busqueda del umbral optimo
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, precision_recall_curve

# 1. Curva de calibracion
prob_true, prob_pred = calibration_curve(y_train, proba_hgb, n_bins=8)
brier = brier_score_loss(y_train, proba_hgb)

plt.figure(figsize=(7, 4.5))
plt.plot(prob_pred, prob_true, marker="o", label=f"HGB (Brier score = {brier:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Calibración perfecta")
plt.title("Curva de Calibración de Probabilidades (Reliability Curve)")
plt.xlabel("Probabilidad media predicha")
plt.ylabel("Fracción real de supervivientes")
plt.legend()
plt.show()

# 2. Busqueda del umbral que maximiza F1
prec, rec, thresholds = precision_recall_curve(y_train, proba_hgb)
f1_scores = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-10)
mejor_idx = np.argmax(f1_scores)
mejor_umbral = thresholds[mejor_idx]

print(f"Brier score: {brier:.4f} (valores cercanos a 0 indican probabilidades confiables)")
print(f"Umbral que maximiza F1: {mejor_umbral:.4f} (F1 alcanzado: {f1_scores[mejor_idx]:.4f})")
print(f"F1 con umbral estandar 0.50: {f1_scores[np.argmin(np.abs(thresholds - 0.50))]:.4f}")

### Paso 54. Análisis de errores por subgrupos y reflexión ética

Desglosamos la tasa de error del modelo por género y por clase social para entender dónde se equivoca más y examinar casos de alta confianza.

In [ ]:
# Paso 54: Diagnostico de errores por subgrupos sociodemograficos
df_analisis = X_train.copy()
df_analisis["Real"] = y_train
df_analisis["Pred"] = pred_hgb
df_analisis["Proba"] = proba_hgb.round(3)
df_analisis["Error"] = (df_analisis["Real"] != df_analisis["Pred"])

print("Tasa de error por Género:")
display(df_analisis.groupby("Sex")["Error"].agg(["mean", "count"]).rename(columns={"mean": "Tasa Error", "count": "Pasajeros"}))

print("\nTasa de error por Clase Social (Pclass):")
display(df_analisis.groupby("Pclass")["Error"].agg(["mean", "count"]).rename(columns={"mean": "Tasa Error", "count": "Pasajeros"}))

# Ejemplos de errores con alta confianza (el modelo estaba seguro pero fallo)
print("\nEjemplos de errores de alta confianza:")
errores_extremos = df_analisis[
    ((df_analisis["Proba"] > 0.80) & (df_analisis["Real"] == 0)) |
    ((df_analisis["Proba"] < 0.20) & (df_analisis["Real"] == 1))
]
display(errores_extremos[["Pclass", "Sex", "Age", "Fare", "TTitulo", "Proba", "Real"]].head(6))


> **Reflexión Ética:**
> El modelo replica con éxito los patrones de supervivencia de 1912, donde el género y la clase social determinaban el acceso a los botes salvavidas. Que el modelo prediga con exactitud *lo que ocurrió históricamente* no significa que *deba ocurrir así*. Este modelo **nunca** debería usarse para guiar protocolos de evacuación modernos (donde el rescate debe basarse en vulnerabilidad física y ubicación sin distinción social) ni para tarificación discriminatoria en seguros contemporáneos.

### Paso 55. Curva de aprendizaje: ¿más datos o un modelo distinto?

Analizamos la curva de aprendizaje para diagnosticar si el modelo sufre de **alto sesgo** (subajuste) o **alta varianza** (sobreajuste):

In [ ]:
# Paso 55: Curva de aprendizaje (Learning Curve) de HistGradientBoosting
from sklearn.model_selection import learning_curve

tamanios, train_scores, val_scores = learning_curve(
    pipe_hgb, X_train, y_train, cv=cv,
    train_sizes=np.linspace(0.2, 1.0, 5), scoring="roc_auc"
)

train_media = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_media = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(8, 4.5))
plt.plot(tamanios, train_media, "o-", color="blue", label="Entrenamiento")
plt.fill_between(tamanios, train_media - train_std, train_media + train_std, alpha=0.15, color="blue")

plt.plot(tamanios, val_media, "o-", color="green", label="Validación cruzada")
plt.fill_between(tamanios, val_media - val_std, val_media + val_std, alpha=0.15, color="green")

plt.title("Curva de Aprendizaje - HistGradientBoosting")
plt.xlabel("Muestras de entrenamiento utilizadas")
plt.ylabel("ROC-AUC")
plt.legend()
plt.show()

print(f"ROC-AUC con 113 muestras: {val_media[0]:.4f}")
print(f"ROC-AUC con 569 muestras: {val_media[-1]:.4f}")
print("Diagnóstico: La curva de validación sigue una trayectoria ascendente y converge cerca de 0.89, demostrando que el modelo aprovecha muy bien los datos adicionales.")

## Parte 14 — Ajuste de hiperparámetros (Pasos 56–58)

Ajustamos los hiperparámetros del modelo con validación cruzada exclusivamente sobre `X_train`. Esto busca la configuración óptima sin incurrir en optimismo falso.

### Paso 56. Definir el espacio de búsqueda

Definimos las rejillas de parámetros en `flows/05_ml_models.py` con el prefijo `model__`:

In [ ]:
# Paso 56: Consultamos las rejillas de busqueda definidas en flows/05_ml_models.py
rejillas = ml.get_param_grids()

print("Rejilla para HistGradientBoosting:")
for k, v in rejillas["hgb"].items():
    print(f"  - {k}: {v}")

print("\nRejilla para Regresión Logística:")
for k, v in rejillas["logreg"].items():
    print(f"  - {k}: {v}")


### Paso 57. Búsqueda con validación cruzada (`GridSearchCV`)

Ejecutamos `GridSearchCV` sobre los 5 pliegues para encontrar la mejor combinación para nuestros dos finalistas:

In [ ]:
# Paso 57: Busqueda en rejilla con validacion cruzada sobre X_train
from sklearn.model_selection import GridSearchCV

# 1. Busqueda para HistGradientBoosting
pipe_hgb_base = ml.build_pipeline(modelos_candidatos["hgb"])
gs_hgb = GridSearchCV(pipe_hgb_base, rejillas["hgb"], scoring="roc_auc", cv=cv, refit=True)
gs_hgb.fit(X_train, y_train)

# 2. Busqueda para Regresion Logistica
pipe_lr_base = ml.build_pipeline(modelos_candidatos["logreg"])
gs_lr = GridSearchCV(pipe_lr_base, rejillas["logreg"], scoring="roc_auc", cv=cv, refit=True)
gs_lr.fit(X_train, y_train)

print(f"HGB - Mejor ROC-AUC: {gs_hgb.best_score_:.4f}")
print(f"HGB - Mejores parámetros: {gs_hgb.best_params_}")
print(f"\nLogReg - Mejor ROC-AUC: {gs_lr.best_score_:.4f}")
print(f"LogReg - Mejores parámetros: {gs_lr.best_params_}")


### Paso 58. ¿Mejoró de verdad? (Comparación pareada)

Comparamos pliegue a pliegue el modelo con parámetros por defecto frente al modelo optimizado:

In [ ]:
# Paso 58: Comparacion pareada pliegue a pliegue
from sklearn.model_selection import cross_validate

res_defecto = cross_validate(pipe_hgb_base, X_train, y_train, cv=cv, scoring="roc_auc")["test_score"]
res_optimizado = cross_validate(gs_hgb.best_estimator_, X_train, y_train, cv=cv, scoring="roc_auc")["test_score"]

diferencias = res_optimizado - res_defecto
df_comparacion = pd.DataFrame({
    "HGB Defecto": res_defecto.round(4),
    "HGB Optimizado": res_optimizado.round(4),
    "Diferencia (Opt - Def)": diferencias.round(4)
})
display(df_comparacion)

print(f"Mejora media en ROC-AUC: {diferencias.mean():+.4f} (desviación: {diferencias.std():.4f})")
print("Conclusión: La optimización mejora ligeramente el puntaje (+0.0058 ROC-AUC), consolidando hiperparámetros estables.")

## Parte 15 — Interpretación y ablación de variables (Pasos 59–61)

Para que un modelo sea confiable en producción (especialmente en seguros), debemos poder explicar qué factores pesan más en su decisión y verificar si todas las variables son necesarias.

### Paso 59. Coeficientes del modelo lineal y Odds Ratios

Inspeccionamos los coeficientes de la Regresión Logística y sus razones de momios (*Odds Ratios* = $\exp(\beta)$):

In [ ]:
# Paso 59: Analisis de coeficientes y razones de momios (Odds Ratios)
pipe_lr_final = gs_lr.best_estimator_.fit(X_train, y_train)
nombres_feat = pipe_lr_final["prep"].get_feature_names_out()
coeficientes = pipe_lr_final["model"].coef_[0]

df_coef = pd.DataFrame({
    "Variable": nombres_feat,
    "Coeficiente": coeficientes.round(4),
    "Odds Ratio (exp)": np.exp(coeficientes).round(4),
    "Impacto": np.where(coeficientes > 0, "Aumenta supervivencia", "Disminuye supervivencia")
}).sort_values("Odds Ratio (exp)", ascending=False)

display(df_coef)
print("\nInterpretación clave:")
print("1. cat__Pclass_1 (Odds Ratio ≈ 3.2): Viajar en 1ª clase multiplica por más de 3 las probabilidades de sobrevivir frente al promedio.")
print("2. cat__Sex_female (Odds Ratio ≈ 2.5): Ser mujer multiplica por 2.5 las probabilidades de sobrevivir.")
print("3. cat__Pclass_3 (Odds Ratio ≈ 0.33): Viajar en 3ª clase reduce las probabilidades de supervivencia a un tercio.")


### Paso 60. Importancia de variables por permutación

Barajamos los valores de cada variable original sobre datos de validación (`X_val`) para medir cuánto cae el ROC-AUC:

In [ ]:
# Paso 60: Importancia por permutacion en datos no vistos de entrenamiento
from sklearn.inspection import permutation_importance

# Particion interna dentro de X_train (sin tocar X_test)
X_tr, X_val, y_tr, y_val = ml.split(X_train, y_train, test_size=0.25, random_state=42)
pipe_hgb_final = gs_hgb.best_estimator_.fit(X_tr, y_tr)

perm = permutation_importance(pipe_hgb_final, X_val, y_val, scoring="roc_auc", n_repeats=20, random_state=42)

df_perm = pd.DataFrame({
    "Variable": X_val.columns,
    "Importancia Media": perm.importances_mean.round(4),
    "Desviación": perm.importances_std.round(4)
}).sort_values("Importancia Media", ascending=False)

plt.figure(figsize=(8, 4.5))
plt.barh(df_perm["Variable"], df_perm["Importancia Media"], xerr=df_perm["Desviación"], color="teal", alpha=0.8)
plt.title("Importancia por Permutación de Variables Originales (HistGradientBoosting)")
plt.xlabel("Caída media en ROC-AUC al barajar")
plt.gca().invert_yaxis()
plt.show()

display(df_perm)

### Paso 61. Estudio de ablación de variables

Entrenamos con y sin diferentes grupos de variables para cuantificar con precisión su aporte:

In [ ]:
# Paso 61: Estudio de ablacion de variables
configuraciones = {
    "A) Base (Numéricas + Categóricas)": (ml.NUM_FEATURES, ml.CAT_FEATURES),
    "B) Base + Cabina (HasCabin, TCabin)": (ml.NUM_FEATURES + ml.CABIN_NUM, ml.CAT_FEATURES + ml.CABIN_CAT),
    "C) Base sin Título social": (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != "TTitulo"]),
    "D) Base sin Género directo": (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != "Sex"]),
}

resultados_ablacion = []
base_auc = None

for nombre_conf, (nf, cf) in configuraciones.items():
    pipe_abl = ml.build_pipeline(gs_hgb.best_estimator_["model"], num_features=nf, cat_features=cf)
    scores = cross_validate(pipe_abl, X_train, y_train, cv=cv, scoring="roc_auc")["test_score"]
    mean_auc = scores.mean()
    if base_auc is None:
        base_auc = mean_auc
    resultados_ablacion.append({
        "Configuración": nombre_conf,
        "ROC-AUC (CV)": f"{mean_auc:.4f} +/- {scores.std():.4f}",
        "Diferencia vs Base": f"{mean_auc - base_auc:+.4f}"
    })

display(pd.DataFrame(resultados_ablacion))
print("\nDecisión final: Las variables de cabina apenas aportan +0.0004 de ROC-AUC y presentan riesgo de sesgo de registro histórico. Por principio de parsimonia y rigor analítico, se excluyen del modelo final.")


## Parte 16 — Evaluación final sobre el conjunto de prueba (Pasos 62–63)

> **Regla de oro — Abrir el conjunto de prueba una sola vez:** Todo el proceso previo (ingeniería de datos, exploración, selección de familias de modelos, calibración y ajuste fino) se realizó exclusivamente sobre `X_train` usando validación cruzada estratificada. Ahora evaluamos el modelo definitivo (`HistGradientBoostingClassifier`) una sola vez sobre `X_test` para obtener una estimación insesgada de generalización. No alteramos ningún parámetro tras ver este resultado.

### Paso 62. Evaluar en prueba con incertidumbre

In [ ]:
# Paso 62: Evaluación honesta sobre X_test y cálculo de intervalos de confianza por Bootstrap
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from flows.common.config import settings

pipe_final = ml.build_pipeline(
    ml.get_candidates(settings.random_state)["hgb"]
).set_params(**ml.FINAL_PARAMS)

pipe_final.fit(X_train, y_train)

# 1. Métrica puntual en Test (única evaluación autorizada)
test_metrics = ml.evaluate_on_test(pipe_final, X_test, y_test, threshold=ml.THRESHOLD)
print(f"Desempeño final en Test (n={test_metrics['n_test']}):")
print(f"  Exactitud (Accuracy): {test_metrics['accuracy']:.4f}")
print(f"  F1-Score:             {test_metrics['f1']:.4f}")
print(f"  ROC-AUC:              {test_metrics['roc_auc']:.4f}")

# 2. Intervalo de confianza al 95% mediante Bootstrap no paramétrico (1000 remuestreos)
rng = np.random.default_rng(settings.random_state)
proba_test = pipe_final.predict_proba(X_test)[:, 1]
n_boot = 1000
boot_aucs, boot_accs, boot_f1s = [], [], []
n_test = len(y_test)

for _ in range(n_boot):
    idx = rng.choice(n_test, size=n_test, replace=True)
    y_b = y_test.iloc[idx]
    if len(np.unique(y_b)) < 2:
        continue
    p_b = proba_test[idx]
    pred_b = (p_b >= ml.THRESHOLD).astype(int)
    boot_aucs.append(roc_auc_score(y_b, p_b))
    boot_accs.append(accuracy_score(y_b, pred_b))
    boot_f1s.append(f1_score(y_b, pred_b))

ci_auc = np.percentile(boot_aucs, [2.5, 97.5])
ci_acc = np.percentile(boot_accs, [2.5, 97.5])
ci_f1 = np.percentile(boot_f1s, [2.5, 97.5])

print("\nIntervalos de Confianza al 95% (Bootstrap):")
print(f"  ROC-AUC:   [{ci_auc[0]:.4f}, {ci_auc[1]:.4f}] (amplitud: {ci_auc[1]-ci_auc[0]:.4f})")
print(f"  Exactitud: [{ci_acc[0]:.4f}, {ci_acc[1]:.4f}]")
print(f"  F1-Score:  [{ci_f1[0]:.4f}, {ci_f1[1]:.4f}]")


### Paso 63. Comparar, contrastar y documentar limitaciones

Comparamos el modelo final frente a los baselines de referencia (`DummyClassifier` y la regla heurística de género) tanto en validación cruzada sobre entrenamiento como en la partición de prueba independiente:

In [ ]:
# Paso 63: Tabla comparativa final de modelos (Train CV vs Test)
gender_pred_test = (X_test["Sex"] == "female").astype(int)
gender_acc_test = accuracy_score(y_test, gender_pred_test)
gender_f1_test = f1_score(y_test, gender_pred_test)
gender_auc_test = roc_auc_score(y_test, gender_pred_test)

dummy_pipe = ml.build_pipeline(ml.get_candidates(settings.random_state)["dummy"]).fit(X_train, y_train)
dummy_acc_test = accuracy_score(y_test, dummy_pipe.predict(X_test))
dummy_f1_test = f1_score(y_test, dummy_pipe.predict(X_test))
dummy_auc_test = roc_auc_score(y_test, dummy_pipe.predict_proba(X_test)[:, 1])

resumen_final = pd.DataFrame([
    {
        "Modelo": "Dummy (Clase Mayoritaria)",
        "CV ROC-AUC": "0.5000",
        "Test ROC-AUC": f"{dummy_auc_test:.4f}",
        "CV Exactitud": "0.6166",
        "Test Exactitud": f"{dummy_acc_test:.4f}",
        "CV F1": "0.0000",
        "Test F1": f"{dummy_f1_test:.4f}",
    },
    {
        "Modelo": "Regla Heurística Género (Mujeres=1)",
        "CV ROC-AUC": "0.7712",
        "Test ROC-AUC": f"{gender_auc_test:.4f}",
        "CV Exactitud": "0.7865",
        "Test Exactitud": f"{gender_acc_test:.4f}",
        "CV F1": "0.7300",
        "Test F1": f"{gender_f1_test:.4f}",
    },
    {
        "Modelo": "HistGradientBoosting (Ajustado)",
        "CV ROC-AUC": "0.8925 ± 0.024",
        "Test ROC-AUC": f"{test_metrics['roc_auc']:.4f} [{ci_auc[0]:.2f}, {ci_auc[1]:.2f}]",
        "CV Exactitud": "0.8357 ± 0.035",
        "Test Exactitud": f"{test_metrics['accuracy']:.4f} [{ci_acc[0]:.2f}, {ci_acc[1]:.2f}]",
        "CV F1": "0.7792 ± 0.043",
        "Test F1": f"{test_metrics['f1']:.4f} [{ci_f1[0]:.2f}, {ci_f1[1]:.2f}]",
    },
])

display(resumen_final)

## Parte 17 — Entrenamiento final y guardado del modelo (Pasos 64–69)

### Paso 64. Congelar las decisiones en el módulo productivo

Las decisiones técnicas y de negocio quedaron formalizadas en constantes inmutables dentro del módulo `flows/05_ml_models.py`:
- **Estimador:** `HistGradientBoostingClassifier`
- **Hiperparámetros:** `learning_rate=0.1, max_depth=4, max_iter=200, l2_regularization=1.0`
- **Variables de cabina:** Excluidas (`USE_CABIN_FEATURES = False`) para no explotar sesgos de registro histórico.
- **Umbral de clasificación:** `0.50`

In [ ]:
# Paso 64: Consultar las decisiones congeladas en flows/05_ml_models.py
print(f"Estimador final:          {ml.FINAL_ESTIMATOR}")
print(f"Hiperparámetros elegidos: {ml.FINAL_PARAMS}")
print(f"Usa variables de cabina:  {ml.USE_CABIN_FEATURES}")
print(f"Umbral operativo:         {ml.THRESHOLD}")
print(f"Columnas numéricas:       {ml.NUM_FEATURES}")
print(f"Columnas categóricas:     {ml.CAT_FEATURES}")

### Pasos 65 y 66. Entrenar con todos los datos y guardar el artefacto

Con el protocolo de validación ya culminado, reentrenamos el modelo final sobre la totalidad de los 891 registros disponibles (`X` e `y` completos) y generamos el binario serializado `.joblib` junto a su archivo de metadatos `.metadata.json` para auditoría y reproducibilidad.

In [ ]:
# Pasos 65 y 66: Reentrenamiento definitivo y persistencia del modelo con metadatos
model_saved_path = ml.run()
print(f"Artefacto serializado con éxito en: {model_saved_path}")

# Cargamos el modelo y verificamos sus metadatos
loaded_pipeline, metadata = ml.load_model(model_saved_path)
print("\nMetadatos del modelo guardado:")
print(f"  Nombre:          {metadata['name']}")
print(f"  Estimador:       {metadata['estimator']}")
print(f"  Filas de train:  {metadata['n_train']}")
print(f"  Versión sklearn: {metadata['sklearn_version']}")
print(f"  SHA-256 modelo:  {metadata['model_sha256'][:16]}...")
print(f"  Métricas test:   {metadata['metrics_test']}")

### Pasos 67 a 69. Prueba de humo con pasajeros no etiquetados

Cargamos los datos sin etiqueta (`data/bronze/titanic_unlabeled.csv`), les aplicamos las transformaciones deterministas de ingeniería de características (`build_silver`), seleccionamos las columnas requeridas por el modelo y ejecutamos `ml.predict()`. Esta prueba de humo verifica que el pipeline corre de extremo a extremo sin errores y que las probabilidades generadas son coherentes.

In [ ]:
# Pasos 67 a 69: Cargar y predecir sobre nuevos pasajeros (Prueba de Humo)
import importlib

from flows.common.paths import BRONZE, BRONZE_UNLABELED_NAME

de = importlib.import_module("flows.01_data_engineering")

# 1. Cargar archivo no etiquetado de Bronze
df_unlabeled_raw = pd.read_csv(BRONZE / BRONZE_UNLABELED_NAME)
print(f"Pasajeros sin etiqueta cargados: {len(df_unlabeled_raw)}")

# 2. Aplicar ingeniería de características determinista de Silver
df_unlabeled_silver = de.build_silver(df_unlabeled_raw)

# 3. Preparar matriz de variables candidatas usando AgeOriginal
X_new = df_unlabeled_silver[ml.CANDIDATE_COLUMNS].copy()
X_new["Age"] = df_unlabeled_silver["AgeOriginal"]

# 4. Generar predicciones y probabilidades con el umbral óptimo
predicciones_nuevas = ml.predict(loaded_pipeline, X_new, threshold=ml.THRESHOLD)

print("\nPrimeras 10 predicciones:")
display(predicciones_nuevas.head(10))

print("\nDiagnóstico de la inferencia:")
print(f"  Total de predicciones: {len(predicciones_nuevas)}")
print(f"  Valores nulos en salida: {predicciones_nuevas.isna().sum().to_dict()}")
print("  Distribución de clases predichas:")
print(predicciones_nuevas["pred"].value_counts(normalize=True).rename({0: "No sobrevive (0)", 1: "Sobrevive (1)"}))

# Visualizar la distribución de probabilidades predichas
plt.figure(figsize=(8, 4))
sns.histplot(predicciones_nuevas["proba_survive"], bins=25, kde=True, color="teal")
plt.axvline(ml.THRESHOLD, color="red", linestyle="--", label=f"Umbral ({ml.THRESHOLD})")
plt.title("Distribución de Probabilidades Predichas (Pasajeros No Etiquetados)")
plt.xlabel("Probabilidad de Supervivencia")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.show()